In [0]:
from pyspark.sql import functions as F

B, S = "workspace.bronze", "workspace.silver"

# --- orders ---
ts_cols = ["order_purchase_timestamp","order_approved_at",
           "order_delivered_carrier_date","order_delivered_customer_date",
           "order_estimated_delivery_date"]
orders = spark.table(f"{B}.orders").dropDuplicates(["order_id"])
for c in ts_cols:
    orders = orders.withColumn(c, F.to_timestamp(c))
orders = (orders
    .withColumn("order_status", F.lower(F.trim("order_status")))
    .withColumn("purchase_date", F.to_date("order_purchase_timestamp"))
    .filter(F.col("order_id").isNotNull() & F.col("customer_id").isNotNull()))
orders.write.format("delta").mode("overwrite").saveAsTable(f"{S}.orders")

# --- customers ---
(spark.table(f"{B}.customers").dropDuplicates(["customer_id"])
   .withColumn("customer_city", F.initcap(F.trim("customer_city")))
   .withColumn("customer_state", F.upper(F.trim("customer_state")))
   .write.format("delta").mode("overwrite").saveAsTable(f"{S}.customers"))

# --- products (null handling + English category names) ---
tr = (spark.table(f"{B}.product_category_name_translation")
        .select("product_category_name", "product_category_name_english"))
prod = (spark.table(f"{B}.products").dropDuplicates(["product_id"])
    .join(tr, "product_category_name", "left")
    .withColumn("category", F.coalesce("product_category_name_english",
                                       "product_category_name", F.lit("unknown")))
    .fillna({"product_weight_g": 0})
    .drop("product_category_name_english"))
prod.write.format("delta").mode("overwrite").saveAsTable(f"{S}.products")

# --- sellers ---
(spark.table(f"{B}.sellers").dropDuplicates(["seller_id"])
   .withColumn("seller_state", F.upper(F.trim("seller_state")))
   .write.format("delta").mode("overwrite").saveAsTable(f"{S}.sellers"))

# --- order_items ---
(spark.table(f"{B}.order_items")
   .dropDuplicates(["order_id","order_item_id"])
   .withColumn("price", F.col("price").cast("double"))
   .withColumn("freight_value", F.col("freight_value").cast("double"))
   .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date"))
   .filter(F.col("price") >= 0)
   .write.format("delta").mode("overwrite").saveAsTable(f"{S}.order_items"))

# --- order_payments ---
(spark.table(f"{B}.order_payments")
   .dropDuplicates(["order_id","payment_sequential"])
   .withColumn("payment_value", F.col("payment_value").cast("double"))
   .withColumn("payment_type", F.lower(F.trim("payment_type")))
   .write.format("delta").mode("overwrite").saveAsTable(f"{S}.order_payments"))

# --- order_reviews (known duplicate review_ids in this dataset) ---
(spark.table(f"{B}.order_reviews").dropDuplicates(["review_id","order_id"])
   .withColumn("review_score", F.col("review_score").cast("int"))
   .withColumn("review_creation_date", F.to_timestamp("review_creation_date"))
   .write.format("delta").mode("overwrite").saveAsTable(f"{S}.order_reviews"))

In [0]:
from pyspark.sql import functions as F

class DQError(Exception):
    pass

def dq_check(table, pk=None, not_null=None, expected_cols=None,
             min_rows=1, fks=None):
    """
    pk: list of columns forming the primary key (uniqueness check)
    not_null: columns that must have no nulls
    expected_cols: columns that must exist (schema validation)
    fks: list of (col, ref_table, ref_col) for referential integrity
    """
    df = spark.table(table)
    results = []

    # row count
    n = df.count()
    results.append(("row_count>=%d" % min_rows, n >= min_rows, n))

    # schema validation
    if expected_cols:
        missing = [c for c in expected_cols if c not in df.columns]
        results.append(("schema_has_expected_cols", not missing, missing))

    # null checks
    for c in (not_null or []):
        nulls = df.filter(F.col(c).isNull()).count()
        results.append((f"not_null:{c}", nulls == 0, nulls))

    # uniqueness
    if pk:
        dups = n - df.select(*pk).distinct().count()
        results.append((f"unique:{'+'.join(pk)}", dups == 0, dups))

    # referential integrity
    for col, ref_table, ref_col in (fks or []):
        ref = spark.table(ref_table).select(F.col(ref_col).alias("_k")).distinct()
        orphans = (df.select(col).where(F.col(col).isNotNull()).distinct()
                     .join(ref, F.col(col) == F.col("_k"), "left_anti").count())
        results.append((f"fk:{col}->{ref_table}.{ref_col}", orphans == 0, orphans))

    failed = [r for r in results if not r[1]]
    print(f"--- DQ {table} ---")
    for name, ok, val in results:
        print(("PASS" if ok else "FAIL"), name, val)
    if failed:
        raise DQError(f"{table}: {len(failed)} check(s) failed")
    return True

In [0]:
dq_check("workspace.silver.orders", pk=["order_id"],
         not_null=["order_id","customer_id","order_purchase_timestamp"],
         fks=[("customer_id","workspace.silver.customers","customer_id")])
dq_check("workspace.silver.customers", pk=["customer_id"], not_null=["customer_id"])
dq_check("workspace.silver.products", pk=["product_id"], not_null=["product_id","category"])
dq_check("workspace.silver.sellers", pk=["seller_id"], not_null=["seller_id"])
dq_check("workspace.silver.order_items", pk=["order_id","order_item_id"],
         not_null=["order_id","product_id","seller_id"],
         fks=[("order_id","workspace.silver.orders","order_id"),
              ("product_id","workspace.silver.products","product_id"),
              ("seller_id","workspace.silver.sellers","seller_id")])
dq_check("workspace.silver.order_payments", pk=["order_id","payment_sequential"],
         fks=[("order_id","workspace.silver.orders","order_id")])

--- DQ workspace.silver.orders ---
PASS row_count>=1 99441
PASS not_null:order_id 0
PASS not_null:customer_id 0
PASS not_null:order_purchase_timestamp 0
PASS unique:order_id 0
PASS fk:customer_id->workspace.silver.customers.customer_id 0
--- DQ workspace.silver.customers ---
PASS row_count>=1 99441
PASS not_null:customer_id 0
PASS unique:customer_id 0
--- DQ workspace.silver.products ---
PASS row_count>=1 32951
PASS not_null:product_id 0
PASS not_null:category 0
PASS unique:product_id 0
--- DQ workspace.silver.sellers ---
PASS row_count>=1 3095
PASS not_null:seller_id 0
PASS unique:seller_id 0
--- DQ workspace.silver.order_items ---
PASS row_count>=1 112650
PASS not_null:order_id 0
PASS not_null:product_id 0
PASS not_null:seller_id 0
PASS unique:order_id+order_item_id 0
PASS fk:order_id->workspace.silver.orders.order_id 0
PASS fk:product_id->workspace.silver.products.product_id 0
PASS fk:seller_id->workspace.silver.sellers.seller_id 0
--- DQ workspace.silver.order_payments ---
PASS row

True

In [0]:
from pyspark.sql import functions as F
S, G = "workspace.silver", "workspace.gold"

# dim_customer
(spark.table(f"{S}.customers")
   .select("customer_id","customer_unique_id","customer_city","customer_state")
   .write.format("delta").mode("overwrite").saveAsTable(f"{G}.dim_customer"))

# dim_product
(spark.table(f"{S}.products")
   .select("product_id","category","product_weight_g",
           "product_length_cm","product_height_cm","product_width_cm")
   .write.format("delta").mode("overwrite").saveAsTable(f"{G}.dim_product"))

# dim_seller
(spark.table(f"{S}.sellers")
   .select("seller_id","seller_city","seller_state")
   .write.format("delta").mode("overwrite").saveAsTable(f"{G}.dim_seller"))

# dim_date (generated calendar)
dates = spark.table(f"{S}.orders").agg(F.min("purchase_date").alias("a"),
                                       F.max("purchase_date").alias("b")).first()
cal = (spark.sql(f"SELECT explode(sequence(to_date('{dates.a}'), to_date('{dates.b}'), interval 1 day)) AS date")
    .withColumn("date_key", F.date_format("date","yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date")).withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date","MMM"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("day_of_week", F.date_format("date","E")))
cal.write.format("delta").mode("overwrite").saveAsTable(f"{G}.dim_date")

# fact_order_items (grain: one row per item in an order)
o = spark.table(f"{S}.orders")
i = spark.table(f"{S}.order_items")
fact = (i.join(o, "order_id")
  .select("order_id","order_item_id","customer_id","product_id","seller_id",
          F.date_format("purchase_date","yyyyMMdd").cast("int").alias("date_key"),
          "order_status","price","freight_value",
          (F.col("price")+F.col("freight_value")).alias("item_total"),
          F.datediff("order_delivered_customer_date","order_purchase_timestamp").alias("delivery_days"),
          F.datediff("order_delivered_customer_date","order_estimated_delivery_date").alias("days_vs_estimate"),
          (F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date")).alias("is_late")))
fact.write.format("delta").mode("overwrite").saveAsTable(f"{G}.fact_order_items")

# fact_payments (grain: one row per payment per order)
(spark.table(f"{S}.order_payments")
   .select("order_id","payment_sequential","payment_type","payment_installments","payment_value")
   .write.format("delta").mode("overwrite").saveAsTable(f"{G}.fact_payments"))